terminal updated version: increase flash size and add IMU

In [ ]:
%%bash
git clone https://github.com/lvgl-micropython/lvgl_micropython.git
cd lvgl_micropython
apt-get update
apt-get install -y gcc g++ make automake python3 git gcc-arm-none-eabi libusb-1.0-0 python3.12-venv cmake python3-pip python3-full
python3 make.py esp32 clean \
  --flash-size=8 \
  BOARD=ESP32_GENERIC_C6 \
  DISPLAY=jd9853 \
  INDEV=axs5106 \
  IMU=qmi8658c

Python version with auto download:

In [1]:
import subprocess, os, re, traceback, datetime
from google.colab import files

REPO_DIR = "/content/lvgl_micropython"
BOARD = "ESP32_GENERIC_C6"
FLASH_SIZE = 8
LOG_PATH = "/content/build_log.txt"
ERROR_REPORT_PATH = "/content/build_error_report.txt"

log_lines = []

# Match common progress bar/refresh patterns, skip printing/recording if matched
PROGRESS_PATTERNS = [
    re.compile(r'^\s*\d+%\|'),                      # tqdm style: 45%|████
    re.compile(r'Progress:\s*\[\s*\d+%\]'),          # apt progress
    re.compile(r'^\s*\d+%\s*$'),                     # pure percentage line
    re.compile(r'\.{3,}\s*\d+%'),                    # ...45%
    re.compile(r'^\s*Downloading\s'),                # download prompt (may refresh repeatedly)
    re.compile(r'^\s*\[=*>?\s*\]'),                  # [====>   ] style progress bar
]

def is_noise(line):
    stripped = line.strip()
    if not stripped:
        return False
    for pat in PROGRESS_PATTERNS:
        if pat.search(stripped):
            return True
    return False

def log(msg):
    print(msg)
    log_lines.append(msg)

def run(cmd, cwd=None, step_name="", env_extra=None):
    log(f"\n{'='*60}\n[{step_name}] >>> {cmd}\n{'='*60}")
    env = os.environ.copy()
    env["DEBIAN_FRONTEND"] = "noninteractive"
    if env_extra:
        env.update(env_extra)

    process = subprocess.Popen(
        cmd, shell=True, cwd=cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        line = line.rstrip("\n")
        # Progress bar/refresh content: only log to file (for debugging), don't print to screen
        if is_noise(line):
            log_lines.append(line)
            continue
        print(line)
        log_lines.append(line)
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f"Step [{step_name}] failed, exit code {process.returncode}\nCommand: {cmd}")

def save_log():
    with open(LOG_PATH, "w") as f:
        f.write("\n".join(log_lines))

def write_error_report(step_name, error):
    with open(ERROR_REPORT_PATH, "w") as f:
        f.write(f"Build Failure Report\n")
        f.write(f"Time: {datetime.datetime.now()}\n")
        f.write(f"Failed Step: {step_name}\n")
        f.write(f"Error: {error}\n")
        f.write(f"\n{'='*60}\nFull Log (last 200 lines, including progress details):\n{'='*60}\n")
        f.write("\n".join(log_lines[-200:]))
    files.download(ERROR_REPORT_PATH)

# ---------------------------------------------------------

current_step = "Initialization"
try:
    current_step = "Cloning repository"
    if not os.path.exists(REPO_DIR):
        run(f"git clone --quiet https://github.com/lvgl-micropython/lvgl_micropython.git {REPO_DIR}", step_name=current_step)
    else:
        log("Repository already exists, skipping clone")

    current_step = "Installing dependencies"
    run("apt-get update -qq", step_name=current_step)
    run("apt-get install -y -qq gcc g++ make automake python3 git gcc-arm-none-eabi "
        "libusb-1.0-0 python3.12-venv cmake python3-pip python3-full", step_name=current_step)

    current_step = "Building firmware"
    run(
        f"python3 make.py esp32 clean "
        f"--flash-size={FLASH_SIZE} "
        f"BOARD={BOARD} "
        f"DISPLAY=jd9853 "
        f"INDEV=axs5106 "
        f"IMU=qmi8658c",
        cwd=REPO_DIR, step_name=current_step
    )

    current_step = "Locating firmware file"
    bin_path = f"{REPO_DIR}/build/lvgl_micropy_{BOARD}-{FLASH_SIZE}.bin"
    if not os.path.exists(bin_path):
        build_dir = f"{REPO_DIR}/build"
        candidates = [os.path.join(build_dir, f) for f in os.listdir(build_dir) if f.endswith(".bin")]
        if not candidates:
            raise FileNotFoundError("No .bin files found in build directory")
        bin_path = max(candidates, key=os.path.getmtime)

    log(f"\n✅ Build successful: {bin_path}")
    save_log()
    files.download(bin_path)
    files.download(LOG_PATH)

except Exception as e:
    log(f"\n❌ Failed at step [{current_step}]: {e}")
    log(traceback.format_exc())
    save_log()
    write_error_report(current_step, e)
    print(f"\nError report generated: {ERROR_REPORT_PATH}")

流式输出内容被截断，只能显示最后 5000 行内容。
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.9 MB/s  0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.1 MB/s  0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 15.1 MB/s  0:00:00
Created wheel for esptool: filename=esptool-4.12.dev3-py3-none-any.whl size=616562 sha256=f2532ee7c293db83a4bd8d93c800873751fb4595277a870353c79ba30f9156e2
Stored in directory: /root/.cache/pip/wheels/db/55/e7/cc65c83ed713de2ddbe7014de042e9b1336802340ab0ea131b
Successfully built esptool
Upgrading pip...
Upgrading setuptools...
Installing Python packages
Constraint file: /root/.espressif/espidf.constraints.v5.5.txt
Requirement files:
- /content/lvgl_micropython/lib/esp-idf/tools/requirements/requirements.core.txt
All done! You can now run:
. ./export.sh
Checking "python3" ...
Python 3.12.13
"python3" has been detected
Activating ESP-IDF 5.5
Setting IDF_PATH to '/content/lvgl_micropython/lib/esp-idf'.
* Checking python version ... 3.12.13
* Checking

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

REF:https://peter.quantr.hk/2025/07/waveshare-esp32-c6-1-47-touch-micropython-lvgl/
If you changed flash size please remember to erase the chip before flash new one, otherwise will have error: The filesystem appears to be corrupted. If you had important data there, you
may want to make a flash snapshot to try to recover it. Otherwise, perform
factory reprogramming of MicroPython firmware (completely erase flash, followed
by firmware programming).